# 06. LLM 파인튜닝 (LLaMA-Factory + Qwen2.5-7B)

**환경:** Google Colab T4 GPU  
**모델:** Qwen2.5-7B-Instruct (한국어 우수)  
**데이터:** 30% 샘플링 (~32,000개)  
**예상 시간:** 6~8시간  

---

## 사전 준비
1. Runtime → Change runtime type → **T4 GPU**
2. 데이터 Google Drive에 업로드 완료

In [ ]:
# 셀 1: LLaMA-Factory 설치 (5분)
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install -e ".[torch,bitsandbytes]" -q
print("✅ LLaMA-Factory 설치 완료")

In [ ]:
# 셀 2: Google Drive 연결
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DATA_PATH = Path('/content/drive/MyDrive/CIVILCOMPLAINT/data')
OUTPUT_PATH = Path('/content/drive/MyDrive/CIVILCOMPLAINT/models/llm_qwen')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(f"✅ 데이터: {DATA_PATH}")
print(f"✅ 출력: {OUTPUT_PATH}")

In [ ]:
# 셀 3: 데이터 로드 + 30% 샘플링
import pandas as pd
import json

train_df = pd.read_parquet(DATA_PATH / 'train_llm.parquet')
val_df = pd.read_parquet(DATA_PATH / 'val_llm.parquet')

print(f"원본 Train: {len(train_df):,}")
print(f"원본 Val: {len(val_df):,}")

# 30% 샘플링
train_df = train_df.sample(frac=0.3, random_state=42).reset_index(drop=True)
val_df = val_df.sample(frac=0.3, random_state=42).reset_index(drop=True)

print(f"\n샘플링 후 Train: {len(train_df):,}")
print(f"샘플링 후 Val: {len(val_df):,}")

In [ ]:
# 셀 4: LLaMA-Factory 형식으로 변환
def convert_to_sharegpt(df):
    data = []
    for _, row in df.iterrows():
        if row['input'] and str(row['input']).strip():
            user_content = f"{row['instruction']}\n\n{row['input']}"
        else:
            user_content = str(row['instruction'])
        
        item = {
            "conversations": [
                {"from": "human", "value": user_content},
                {"from": "gpt", "value": str(row['output'])}
            ]
        }
        data.append(item)
    return data

train_data = convert_to_sharegpt(train_df)
val_data = convert_to_sharegpt(val_df)

# 저장
with open('/content/LLaMA-Factory/data/civil_train.json', 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False)

with open('/content/LLaMA-Factory/data/civil_val.json', 'w', encoding='utf-8') as f:
    json.dump(val_data, f, ensure_ascii=False)

print(f"✅ 데이터 변환 완료: {len(train_data):,}개")

In [ ]:
# 셀 5: 데이터셋 등록
dataset_info_path = '/content/LLaMA-Factory/data/dataset_info.json'

with open(dataset_info_path, 'r') as f:
    dataset_info = json.load(f)

dataset_info['civil_complaint'] = {
    "file_name": "civil_train.json",
    "formatting": "sharegpt",
    "columns": {"messages": "conversations"},
    "tags": {
        "role_tag": "from",
        "content_tag": "value",
        "user_tag": "human",
        "assistant_tag": "gpt"
    }
}

with open(dataset_info_path, 'w') as f:
    json.dump(dataset_info, f, indent=2)

print("✅ 데이터셋 등록 완료")

In [ ]:
# 셀 6: 학습 설정 파일 생성
import yaml

config = {
    # 모델
    "model_name_or_path": "Qwen/Qwen2.5-7B-Instruct",
    "trust_remote_code": True,
    
    # 학습 방식
    "stage": "sft",
    "do_train": True,
    "finetuning_type": "lora",
    
    # 데이터
    "dataset": "civil_complaint",
    "template": "qwen",
    "cutoff_len": 512,
    "preprocessing_num_workers": 1,
    
    # 출력
    "output_dir": "/content/output",
    "logging_steps": 50,
    "save_steps": 500,
    "overwrite_output_dir": True,
    
    # 학습 파라미터
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "num_train_epochs": 1,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.1,
    
    # LoRA
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target": "q_proj,v_proj",
    
    # 메모리 최적화
    "quantization_bit": 4,
    "gradient_checkpointing": True,
    
    # 기타
    "fp16": True,
    "report_to": "none",
}

with open('/content/LLaMA-Factory/train_config.yaml', 'w') as f:
    yaml.dump(config, f, allow_unicode=True)

print("✅ 학습 설정 완료")
print(f"   모델: Qwen2.5-7B-Instruct")
print(f"   LoRA rank: 16")
print(f"   배치: 2 x 8 = 16")
print(f"   4bit 양자화: O")

In [ ]:
# 셀 7: GPU 확인
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
torch.cuda.empty_cache()

In [ ]:
# 셀 8: 학습 실행 🚀
print("=" * 60)
print("🚀 파인튜닝 시작")
print("=" * 60)
print(f"모델: Qwen2.5-7B-Instruct (4bit)")
print(f"데이터: {len(train_data):,}개 (30% 샘플)")
print(f"예상 시간: 6~8시간")
print("=" * 60)
print("\n💡 Tip: 다른 탭에서 다른 작업 해도 됨!\n")

!llamafactory-cli train train_config.yaml

In [ ]:
# 셀 9: 학습 완료 후 - Drive에 저장
import shutil

print("✅ 학습 완료! Drive에 저장 중...")

src = Path('/content/output')
dst = OUTPUT_PATH / 'civil_complaint_qwen_lora'

if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)

# 설정 저장
training_info = {
    "base_model": "Qwen/Qwen2.5-7B-Instruct",
    "framework": "LLaMA-Factory",
    "lora_rank": 16,
    "train_samples": len(train_data),
    "sampling_ratio": "30%",
    "quantization": "4bit"
}

with open(dst / 'training_info.json', 'w') as f:
    json.dump(training_info, f, indent=2)

print(f"✅ 저장 완료: {dst}")

In [ ]:
# 셀 10: 추론 테스트
print("=" * 60)
print("🧪 추론 테스트")
print("=" * 60)

# 추론 설정
infer_config = {
    "model_name_or_path": "Qwen/Qwen2.5-7B-Instruct",
    "adapter_name_or_path": "/content/output",
    "template": "qwen",
    "finetuning_type": "lora",
    "quantization_bit": 4,
}

with open('/content/LLaMA-Factory/infer_config.yaml', 'w') as f:
    yaml.dump(infer_config, f)

# CLI로 테스트
!echo "인터넷뱅킹 비밀번호를 잊어버렸어요. 어떻게 해야 하나요?" | llamafactory-cli chat infer_config.yaml

In [ ]:
# 셀 11: 다운로드
!cd /content/drive/MyDrive/CIVILCOMPLAINT/models/llm_qwen && zip -r /content/qwen_lora.zip civil_complaint_qwen_lora

from google.colab import files
files.download('/content/qwen_lora.zip')

print("\n" + "=" * 60)
print("✅ 완료!")
print("=" * 60)
print("다운로드: qwen_lora.zip")
print("사용법: Ollama 또는 vLLM으로 로드")

---
## 완료 후 로컬에서 사용법

### 방법 1: LLaMA-Factory로 추론
```bash
llamafactory-cli chat \
  --model_name_or_path Qwen/Qwen2.5-7B-Instruct \
  --adapter_name_or_path ./civil_complaint_qwen_lora \
  --template qwen \
  --finetuning_type lora
```

### 방법 2: GGUF 변환 후 Ollama
```bash
# 1. 병합
llamafactory-cli export \
  --model_name_or_path Qwen/Qwen2.5-7B-Instruct \
  --adapter_name_or_path ./civil_complaint_qwen_lora \
  --export_dir ./merged_model

# 2. GGUF 변환 (llama.cpp 필요)
python convert.py ./merged_model --outtype q4_k_m

# 3. Ollama에 등록
ollama create civil-qwen -f Modelfile
```